# 05 — Dependencies & Dependency Injection

## Objectives

- Declare a **type** of dependencies on the agent, provide the **instance** per run
- Read deps inside tools via `ctx.deps`
- Add runtime context to prompts with dynamic `@agent.instructions`
- Understand *why*: testability, and per-request state in the FastAPI phase

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

## The problem

Real agents don't run in a vacuum: they need "who is the current user", "which DB connection",
"which HTTP client". Globals and module-level state are the classic hack — and they're
untestable and per-request-unsafe.

Pydantic AI's answer is plain DI:

- `deps_type=MyDeps` on the **Agent** — the *type* (a dataclass is the idiom)
- `deps=MyDeps(...)` on **`run()`** — the *instance*, per run
- `ctx.deps` inside every `@agent.tool` — typed access

Notebook 04's `RunContext[None]` was the empty case. Now we fill it.

In [ ]:
from dataclasses import dataclass

from pydantic_ai import Agent, RunContext


@dataclass
class BankDeps:
    customer_name: str
    balance: float
    currency: str = "USD"


bank_agent = Agent(
    MODEL,
    deps_type=BankDeps,
    instructions="You are helpful bank support. Look up balances with your tool; never guess them.",
)


@bank_agent.tool
async def get_balance(ctx: RunContext[BankDeps]) -> str:
    """Get the current customer's balance, formatted with their currency."""
    return f"{ctx.deps.balance:.2f} {ctx.deps.currency}"


for name, balance in [("Anne", 1250.50), ("Bob", 42.00)]:
    result = await bank_agent.run("How much do I have right now?", deps=BankDeps(name, balance))
    print(f"{name} → {result.output}")

> ✅ **Checkpoint** — same agent, same tools, two different answers: the *deps* changed,
> nothing else did. That's dependency injection doing its job.

> ⚠️ Note: deps are **per-run** — they do **not** ride along in `message_history`. History
> carries the conversation; deps carry the world. Keep both straight and Phase 2 will feel obvious.

## Dynamic instructions

Static `instructions=` is a string you fix at build time. Dynamic instructions are a function
of the run's context — decorated with `@agent.instructions`, receiving `ctx`:

In [ ]:
@bank_agent.instructions
def greet_by_name(ctx: RunContext[BankDeps]) -> str:
    return f"The customer's name is {ctx.deps.customer_name}. Address them by name."


r = await bank_agent.run("Do I have enough for a $60 game?", deps=BankDeps("Anne", 1250.50))
print(r.output)

> ✅ **Checkpoint** — the reply addresses Anne by name, every time, without you writing
> "the customer is Anne" into the prompt.

## Why bother? (the two payoffs)

1. **Testability** — notebook 08 will hand an agent *fake* deps and a fake model, and prove
   tool behavior with zero network and zero cost. With globals, that's impossible.
2. **Servers** — in the FastAPI phase, each HTTP request builds fresh deps (user, DB session,
   HTTP client) and hands them to the same shared agent. You just learned the exact pattern.

## 🏋️ Exercise — deps carrying an HTTP client

Define `@dataclass class WeatherDeps: client: httpx.AsyncClient`. Build a weather agent whose
tools use `ctx.deps.client` to call the **free, keyless** [Open-Meteo API](https://open-meteo.com):

- geocode a city: `https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1`
- get current temperature:
  `https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m`

Then ask: **"Is it warmer right now: Paris or Tokyo?"** — the model must geocode *both* cities
and fetch *both* forecasts through your tools. Raise `ModelRetry` on any request that fails.

Your take: [`../solutions/05_httpx_deps.ipynb`](../solutions/05_httpx_deps.ipynb).

## Next

→ **[06_logfire_observability.ipynb](06_logfire_observability.ipynb)** — now that agents do
real work, learn to *see* them work.